In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

# from datetime import datetime, timedelta

spark = ((SparkSession.builder \
            .appName("spark_joins")
            .master("local[4]")
            .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties")
            .config("spark.log.level", "ERROR")
            .config("spark.eventLog.enabled", "false")
            .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
            
            # Configuração do catálogo Iceberg com Nessie (principal)
            .config("spark.sql.catalog.icebergdw", "org.apache.iceberg.spark.SparkCatalog")
            .config("spark.sql.catalog.icebergdw.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
            .config("spark.sql.catalog.icebergdw.uri", "http://172.19.192.1:19120/api/v1") 
            .config("spark.sql.catalog.icebergdw.ref", "dev") 
            .config("spark.sql.catalog.icebergdw.warehouse", "s3a://iceberg/warehouse/") 
            .config("spark.sql.catalog.icebergdw.default-namespace", "icebergdw")
                
            # Configuração de cache (opcional, se necessário)
            .config("spark.sql.catalog.icebergdw.cache-enabled", "true")
            
            # Dependências (JARs)
            .config("spark.jars.packages", 
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
                    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.13:0.102.5")
            
            # Configurações do MinIO (S3A)
            .config("spark.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
            .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
            .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
            .config("spark.hadoop.fs.s3a.path.style.access", "true")
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
            .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
            
            # Configurações de desempenho
            .config("spark.driver.maxResultSize", "8g")
            .config("spark.executor.memory", "6g")
            .config("spark.executor.cores", "4")
            .config("spark.driver.memory", "18g")
        .getOrCreate()))
  
sc = spark.sparkContext
spark

In [ ]:
dataframe = spark.sql(""" select 
                            row_number() over(order by businessentityid asc) as salespersonkey,
                            * 
                        from icebergdw.advworks_rawdb.raw_dimsalesperson """)

In [ ]:
dataframe = (dataframe
                .withColumn("gender",
                                when((col("gender")).like("M"), "Male")
                                .otherwise("Female")))

In [ ]:
dataframe = (dataframe.withColumn(
                        "territoryid",
                        when(
                            (col("TerritoryID").isNull()) & (col("CountryRegionCode").isNull()) & (col("JobTitle").like("North American%")),
                            lit(1))
                        .when(
                            (col("TerritoryID").isNull()) & (col("CountryRegionCode").isNull()) & (col("JobTitle").like("Pacific Sales%")),
                            lit(9))
                        .when(
                            (col("TerritoryID").isNull()) & (col("CountryRegionCode").isNull()) & (col("JobTitle").like("European Sales%")),
                            lit(8))
                        .otherwise(col("territoryid"))
                    ))

In [ ]:
dataframe = (dataframe.withColumn(
                        "countryregioncode",
                        when(
                            (col("TerritoryID").isNotNull()) & (col("JobTitle").like("North American%")),
                            lit("US"))
                        .when(
                            (col("TerritoryID").isNotNull()) & (col("JobTitle").like("European Sales%")),
                            lit("EU"))
                        .when(
                            (col("TerritoryID").isNotNull()) & (col("JobTitle").like("Pacific Sales%")),
                            lit("UA"))
                        .otherwise(col("countryregioncode"))
                    ))

In [ ]:
dataframe = (dataframe.withColumn(
                        "regionname",
                        when(
                            (col("JobTitle").like("North American%")) & (col("ContinentName").isNull()),
                            lit("USA HQ"))
                        .when(
                            (col("JobTitle").like("Pacific Sales%")) & (col("ContinentName").isNull()),
                            lit("Sidney HQ"))
                        .when(
                            (col("JobTitle").like("European Sales%")) & (col("ContinentName").isNull()),
                            lit("Berlim HQ"))
                        .otherwise(col("ContinentName"))
                    ))

In [ ]:
max_sales_quota = dataframe.selectExpr("cast(max(salesquota) as float)").collect()[0][0]

dataframe = (dataframe
                .withColumn(
                        "salesquota",
                        when((col("jobtitle").like("North American%")) & (col("salesquota").isNull()),
                                lit(max_sales_quota * 0.70 + max_sales_quota))

                        .when((col("jobtitle").like("European Sales%")) & (col("salesquota").isNull()),
                                lit(max_sales_quota * 0.65 + max_sales_quota))

                        .when((col("jobtitle").like("Pacific Sales%")) & (col("salesquota").isNull()),
                                lit(max_sales_quota * 0.55 + max_sales_quota))

                        .otherwise(col("salesquota"))
                    ))

In [ ]:
dataframe = (dataframe 
                .withColumn(
                        "commissionpct",
                        
                         when((col("jobtitle").like('North American%')) & (col("commissionpct")== 0.00), lit(0.05))
                        .when((col("jobtitle").like('European Sales%')) & (col("commissionpct")== 0.00), lit(0.04))                
                        .when((col("jobtitle").like('Pacific Sales%')) & (col("commissionpct")== 0.00), lit(0.04))
                        .otherwise(col("commissionpct"))
                    ))

In [ ]:
dataframe = dataframe.selectExpr("*",
                                 
                                """ 
                                case
                                    when SalariedFlag = 1 then 'Yes'
                                    else 'No'
                                end as issalaried
                                
                                """,
                                
                                """
                                case 
                                    when CurrentFlag = 1 then 'Yes'
                                    else 'No'
                                end as isactive
                                
                                """
                            )

In [ ]:
dataframe = (dataframe
                .withColumn("age", year(current_date()) - year(col("birthdate"))) 
                
                .withColumn("yearsincompany", year(current_date()) - year(col("hiredate")))
            )

In [ ]:
dataframe = (dataframe
                .withColumn("maritalstatus",
                             
                    when((col("maritalstatus")).like("M"), "Married")
                    .otherwise("Single"))
            )

In [ ]:
dataframe = (dataframe
                .withColumn("continentname",
                            when((col("regionname").like("USA HQ")) & (col("continentname").isNull()), "North America")
                            .when((col("regionname").like("Sidney HQ")) & (col("continentname").isNull()), "Pacific")
                            .when((col("regionname").like("Berlim HQ")) & (col("continentname").isNull()), "Europe")
                        .otherwise(col("continentname"))
    ))

In [ ]:
dataframe = (dataframe
                .withColumn("enddate", 
                                when((col("enddate").isNull()) & (col("isactive") == "Yes"), "2100-12-31")
                                .otherwise(col("enddate"))
                            ))

In [ ]:
dataframe = (dataframe
                .withColumn("loginid", 
                            split(col("loginid"),  r"\\")[1]))

In [ ]:
dataframe.drop((col("salariedflag"))).drop((col("currentflag")))

In [ ]:
dataframe = dataframe.select(
                col("salespersonkey"),
                col("businessentityid").alias("salespersonid"),
                col("loginid"),
                col("firstname"),
                col("lastname"),
                col("gender"),
                col("birthdate"),
                col("hiredate"),
                col("age"),
                col("jobtitle"),
                col("maritalstatus"),
                col("organizationlevel"),
                col("departmentid"),
                col("startdate"),
                col("enddate"),
                col("yearsincompany"),
                col("groupname"),
                col("departmentname").alias("department"),
                col("payfrequency"),
                col("rate").cast(DecimalType(5, 2)),
                col("ratechangedate"),
                col("commissionpct").alias("commision").cast(DecimalType(5,2)),
                col("salesquota").alias("salespersonquota").cast(DecimalType(12,2)),
                col("bonus").cast(DecimalType(12,2)),
                col("territoryid"),
                col("regionname").alias("salesregion"),
                col("continentname").alias("continent"),
                col("countryregioncode").alias("countrycode"),
                col("salariedflag"),
                col("currentflag"),
                col("issalaried"),
                col("isactive"),
                col("inserted_date"),
                col("modified_date"),
                col("eltversion")
        )

In [ ]:
dataframe.show(truncate= False)